
# Лекция 10. Нейронные сети и эмбединги

In [1]:
import os
import numpy as np
import random

import matplotlib_inline
import matplotlib.pyplot as plt

%matplotlib inline


np.random.seed(0)
random.seed(0)

## Классификация

Мы хотим использовать векторные представления слов для алгоритмов машинного обучения или нейронных сетей. Прежде чем перейти к нейронным сетям, нам нужны бейзлайны, то есть то, с чем будем сравниваться. Сначала проверим, как работают стандартные алгоритмы машинного обучения на эмбеддингах.

Мы будем решать задачу бинарной классификации для [датасета рецензий с Imdb](https://ai.stanford.edu/~amaas/data/sentiment/).

В качестве базовой модели будем использовать [CatBoost](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier).

Подробный список классических датасетов для классификации текста можно посмотреть [здесь](https://lena-voita.github.io/nlp_course/text_classification.html#dataset_examples)

In [2]:
if not os.path.exists("aclImdb_v1.tar.gz"):
    !wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !tar -xf aclImdb_v1.tar.gz

Мы не будем усердно заниматься предобработкой, так как наша цель — именно сравнить методы, а не добиться более высокого качества. В качестве предобработки мы приведем все символы к нижнему регистру.

In [8]:
def preprocess(l):
    return l.lower()

In [9]:
def get_dataset(split='train', base_dir='aclImdb'):
    path_data   = os.path.join(base_dir, split)
    path_data_n = os.path.join(path_data, 'neg')
    path_data_p = os.path.join(path_data, 'pos')
    path_neg  = list(os.listdir(path_data_n))
    path_neg  = sorted(path_neg)
    path_pos  = list(os.listdir(path_data_p))
    path_pos  = sorted(path_pos)


    print(f"Объектов отрицательного класса {len(path_neg)}")
    print(f"Объектов положительного класса {len(path_pos)}")

    X = []
    y = [0] * len(path_neg) + [1] * len(path_pos)

    for p in path_neg:
        with open(os.path.join(path_data_n, p), 'r') as f:
            l = f.readline()
            X.append(preprocess(l))

    for p in path_pos:
        with open(os.path.join(path_data_p, p), 'r') as f:
            l = f.readline()
            X.append(preprocess(l))

    return X, y

In [10]:
X_train, y_train = get_dataset(split='train')

Объектов отрицательного класса 12500
Объектов положительного класса 12500


In [11]:
X_test, y_test = get_dataset(split='test')

Объектов отрицательного класса 12500
Объектов положительного класса 12500


Посмотрим на примеры предложений.

In [12]:
X_train[0]

"story of a man who has unnatural feelings for a pig. starts out with a opening scene that is a terrific example of absurd comedy. a formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. unfortunately it stays absurd the whole time with no general narrative eventually making it just too off putting. even those from the era should be turned off. the cryptic dialogue would make shakespeare seem easy to a third grader. on a technical level it's better than you might think with some good cinematography by future great vilmos zsigmond. future stars sally kirkland and frederic forrest can be seen briefly."

In [13]:
X_test[0]

"once again mr. costner has dragged out a movie for far longer than necessary. aside from the terrific sea rescue sequences, of which there are very few i just did not care about any of the characters. most of us have ghosts in the closet, and costner's character are realized early on, and then forgotten until much later, by which time i did not care. the character we should really care about is a very cocky, overconfident ashton kutcher. the problem is he comes off as kid who thinks he's better than anyone else around him and shows no signs of a cluttered closet. his only obstacle appears to be winning over costner. finally when we are well past the half way point of this stinker, costner tells us all about kutcher's ghosts. we are told why kutcher is driven to be the best with no prior inkling or foreshadowing. no magic here, it was all i could do to keep from turning it off an hour in."

Для обучения моделей эмбедингов, нам нужно разделить слова на структурные единицы (символы, слова, N-Gramm). Мы будем работать на уровне слов.

Теперь у нас есть **два бейзлайна**:
- `word2vec` + `CatBoost` - accuracy 0.79
- `fasttext` + `CatBoost` - accuracy 0.85

## `Нейронный подход`

Далее мы обучим нейронные сети для классификации текстов. Мы рассмотрим, как токенизировать тексты с помощью библиотеки `tokenizers`, чтобы работа с токенами происходила независимо от архитектуры и обучения. Также мы сравним различные архитектуры для классификации, в частности, MLP и свёрточные.

Важной частью будет сравнение результатов с подходами из классического машинного обучения.

In [ ]:
!pip install tokenizers

In [ ]:
!pip install wandb

In [16]:
from tqdm import tqdm
from functools import partial

import wandb

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import tokenizers
from tokenizers import Tokenizer, trainers

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [17]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def print_params_count(model):

    total_params = sum(p.numel() for p in model.parameters())
    total_params_grad = sum(p.numel() for p in model.parameters() if p.requires_grad)

    model_name = model.__class__.__name__
    print(f"Информация о числе параметров модели: {model_name}")
    print(f"Всего параметров: \t\t {total_params}")
    print(f"Всего обучаемых параметров: \t {total_params_grad}")
    print()

In [18]:
set_global_seed(42)

### `tokenizers`

Прежде чем создавать модель, нам необходимо создать словарь и произвести токенизацию предложений. Именно от словаря будет зависеть, насколько много "слов" знает наша модель. Мы воспользуемся библиотекой [tokenizers](https://huggingface.co/docs/tokenizers/index). Библиотека `tokenizers` является частью инфраструктуры `Hugging Face`.

Токенезация происходит через класс `tokenizer`. Для того чтобы получить `tokenizer` его надо сначала **обучить**, для этого нам необходимо использовать `tokenizers.trainers`. Так как мы будем работать на уровне слов, то выберем `tokenizers.trainers.WordLevelTrainer`.

Для работы с текстами нам необходимо зарезервировать два специальных токена:
1. `<pad>` для токена означающего паддинг
2. `<unk>` для токенов, которые отсутствуют в словаре

#### `WordLevelTrainer`

Для начала мы будем разбивать предложение по словам, для этого воспользуемся `trainers.WordLevelTrainer`. Будем рассматривать словарь размером 5000 слов.

```python
trainers.WordLevelTrainer(self, /, *args, **kwargs)
Docstring:     
Trainer capable of training a WorldLevel model

Args:
    vocab_size (:obj:`int`, `optional`):
        The size of the final vocabulary, including all tokens and alphabet.

    min_frequency (:obj:`int`, `optional`):
        The minimum frequency a pair should have in order to be merged.

    show_progress (:obj:`bool`, `optional`):
        Whether to show progress bars while training.

    special_tokens (:obj:`List[Union[str, AddedToken]]`):
        A list of special tokens the model should know of.
```

In [19]:
top_n_words = 5_000

In [20]:
trainer = trainers.WordLevelTrainer(
    vocab_size     = top_n_words,
    special_tokens = ["<pad>", "<unk>"],
    show_progress  = True
)

tokenizer = tokenizers.Tokenizer(
    model=tokenizers.models.WordLevel()
)

Мы будем обучаться из памяти, поэтому нам необходимо передать итератор, который пройдет по всем текстам, которые уже разбиты на слова.

In [22]:
def get_corpus(X_data):

    return list(x.split() for x in X_data)

In [23]:
corpus_train = get_corpus(X_train)
corpus_test  = get_corpus(X_test)

In [24]:
text_data = corpus_train + corpus_test
text_data[0][:10]

['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a']

In [25]:
tokenizer.train_from_iterator(tqdm(text_data, total=len(text_data)), trainer=trainer)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50000/50000 [00:13<00:00, 3714.20it/s]


Посмотрим на словарь.

In [26]:
print(f"Размер словаря {tokenizer.get_vocab_size()}")

print(list(tokenizer.get_vocab().keys())[:10])

for idx in [0, 1, 2, 21, 42]:
    print(f"idx = {idx}, word = {tokenizer.id_to_token(idx)}")

Размер словаря 5000
['treat', 'bought', 'run', 'going', 'stuff.', 'lawyer', 'life.<br', 'shape', 'forget', 'physical']
idx = 0, word = <pad>
idx = 1, word = <unk>
idx = 2, word = the
idx = 21, word = are
idx = 42, word = about


**Важно:** Cпециальные токены имеют наименьшие *id* по умолчанию.

Для кодирования предложений используется метод `encode`, так как мы самостоятельно описали функцию `tokenize`, то установим `is_pretokenized=True`

In [27]:
tokinezed_text = text_data[0][:20]

print("Исходный текст:")
print(tokinezed_text)

result = tokenizer.encode(tokinezed_text, is_pretokenized=True)
print(result)

print("Tokens ids: ", result.ids)

Исходный текст:
['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig.', 'starts', 'out', 'with', 'a', 'opening', 'scene', 'that', 'is', 'a']
Encoding(num_tokens=20, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
Tokens ids:  [77, 5, 3, 160, 36, 41, 1, 1572, 17, 3, 1, 478, 47, 16, 3, 557, 147, 11, 7, 3]


Для декодирования воспользуемся методом `decode`, мы хотим посмотреть на сколько словарь из 5 тысяч слов выразительный, то есть какие слова мы пропускаем и понятен ли смысл предложения без них.

In [28]:
print("Исходный текст:")
print(tokinezed_text)

print("Результат декодирования:")
print(tokenizer.decode(result.ids, skip_special_tokens=False))

Исходный текст:
['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig.', 'starts', 'out', 'with', 'a', 'opening', 'scene', 'that', 'is', 'a']
Результат декодирования:
story of a man who has <unk> feelings for a <unk> starts out with a opening scene that is a


В примере выше мы видим, что в нашем словаре нет слова `unnatural` и слова `pig.`. Возможно, если бы мы лучше предобработали наш текст, то таких проблем не возникло бы.

### `Данные`

#### `Датасет`

В библиотеке `Hugging Face` существуют различные модули, которые помогают оборачивать токенизаторы в удобные датасеты. Такой подход позволяет описать обучение сложных моделей в 5-10 строчек кода. В этом ноутбуке мы этим и воспользуемся.

Часто бывает полезно ограничить длину предложений параметром `max_len`, чтобы подавать в модель не слишком длинные тексты.

In [29]:
class TextDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len=20):
        super().__init__()

        self.max_len   = max_len
        self.texts     = texts
        self.targets   = targets
        self.tokenizer = tokenizer
        self.tokens    = []

        for text in tqdm(texts):
            tokens = self.tokenizer.encode(text, is_pretokenized=True).ids
            self.tokens.append(tokens)

    def __getitem__(self, idx):
        """
        :param int idx: index of object in dataset
        :return dict: Dictionary with all useful object data
            {
                'text' str: unprocessed text,
                'label' torch.Tensor(dtype=torch.long): sentiment of the text (0 for negative, 1 for positive)
                'rating' torch.Tensor(dtype=torch.long): rating of the text
                'tokens' torch.Tensor(dtype=torch.long): tensor of tokens ids for the text
                'tokens_len' torch.Tensor(dtype=torch.long): number of tokens
            }
        """

        tokens = self.tokens[idx]
        tokens = tokens[:self.max_len]

        return {
            'text': self.texts[idx],
            'target': torch.tensor(self.targets[idx], dtype=torch.long),
            'tokens': torch.tensor(tokens, dtype=torch.long),
            'tokens_len': torch.tensor(len(tokens), dtype=torch.long),
        }

    def __len__(self):
        """
        :return int: number of objects in dataset
        """
        return len(self.targets)

In [30]:
train_dataset = TextDataset(corpus_train, y_train, tokenizer)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25000/25000 [00:10<00:00, 2299.94it/s]


Посмотрим на данные.

In [31]:
train_dataset[0]

{'text': ['story',
  'of',
  'a',
  'man',
  'who',
  'has',
  'unnatural',
  'feelings',
  'for',
  'a',
  'pig.',
  'starts',
  'out',
  'with',
  'a',
  'opening',
  'scene',
  'that',
  'is',
  'a',
  'terrific',
  'example',
  'of',
  'absurd',
  'comedy.',
  'a',
  'formal',
  'orchestra',
  'audience',
  'is',
  'turned',
  'into',
  'an',
  'insane,',
  'violent',
  'mob',
  'by',
  'the',
  'crazy',
  'chantings',
  'of',
  "it's",
  'singers.',
  'unfortunately',
  'it',
  'stays',
  'absurd',
  'the',
  'whole',
  'time',
  'with',
  'no',
  'general',
  'narrative',
  'eventually',
  'making',
  'it',
  'just',
  'too',
  'off',
  'putting.',
  'even',
  'those',
  'from',
  'the',
  'era',
  'should',
  'be',
  'turned',
  'off.',
  'the',
  'cryptic',
  'dialogue',
  'would',
  'make',
  'shakespeare',
  'seem',
  'easy',
  'to',
  'a',
  'third',
  'grader.',
  'on',
  'a',
  'technical',
  'level',
  "it's",
  'better',
  'than',
  'you',
  'might',
  'think',
  'with',

In [32]:
train_dataset[-1]

{'text': ['working-class',
  'romantic',
  'drama',
  'from',
  'director',
  'martin',
  'ritt',
  'is',
  'as',
  'unbelievable',
  'as',
  'they',
  'come,',
  'yet',
  'there',
  'are',
  'moments',
  'of',
  'pleasure',
  'due',
  'mostly',
  'to',
  'the',
  'charisma',
  'of',
  'stars',
  'jane',
  'fonda',
  'and',
  'robert',
  'de',
  'niro',
  '(both',
  'terrific).',
  "she's",
  'a',
  'widow',
  'who',
  "can't",
  'move',
  'on,',
  "he's",
  'illiterate',
  'and',
  'a',
  'closet-inventor--you',
  'can',
  'probably',
  'guess',
  'the',
  'rest.',
  'adaptation',
  'of',
  'pat',
  "barker's",
  'novel',
  '"union',
  'street"',
  '(a',
  'better',
  'title!)',
  'is',
  'so',
  'laid-back',
  'it',
  'verges',
  'on',
  'bland,',
  'and',
  'the',
  "film's",
  'editing',
  'is',
  'a',
  'mess,',
  'but',
  "it's",
  'still',
  'pleasant;',
  'a',
  'rosy-hued',
  'blue-collar',
  'fantasy.',
  'there',
  'are',
  'no',
  'overtures',
  'to',
  'serious',
  'issues

#### `Даталоадер`

Чтобы объединить несколько тензоров разной длины в один можно использовать функцию `torch.nn.utils.rnn.pad_sequence`

Обратите внимание на её аргументы:
1. `batch_first` определяет по какой оси "складывать" тензоры. Предпочтительнее использовать `batch_first=False` так как это может упростить выполнение задания в дальнейшем
2. `padding_value` - число, которое будет использоваться в качестве паддинга, чтобы сделать все тензоры одинаковой длины

In [33]:
torch.nn.utils.rnn.pad_sequence([
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5]),
    torch.tensor([6, 7, 8, 9])
], batch_first=True, padding_value=-1)

tensor([[ 1,  2,  3, -1],
        [ 4,  5, -1, -1],
        [ 6,  7,  8,  9]])

In [34]:
def collate_fn(batch, padding_value, batch_first=True):
    """
    :param List[Dict] batch: List of objects from dataset
    :param int padding_value: Value that will be used to pad tokens
    :param bool batch_first: If True resulting tensor with tokens must have shape [B, T] otherwise [T, B]
    :return dict: Dictionary with all data collated
        {
            'ratings' torch.Tensor(dtype=torch.long): rating of the text for each object in batch
            'labels' to rch.Tensor(dtype=torch.long): sentiment of the text for each object in batch

            'texts' List[str]: All texts in one list
            'tokens' torch.Tensor(dtype=torch.long): tensor of tokens ids padded with @padding_value
            'tokens_lens' torch.Tensor(dtype=torch.long): number of tokens for each object in batch
        }
    """

    texts       = [obj['text'] for obj in batch]
    targets     = torch.stack([obj['target'] for obj in batch])
    tokens_lens = torch.stack([obj['tokens_len'] for obj in batch])
    tokens      = torch.nn.utils.rnn.pad_sequence(
        [obj['tokens'] for obj in batch], batch_first=batch_first, padding_value=padding_value
    )

    return {
        'targets': targets,

        'texts': texts,
        'tokens': tokens,
        'tokens_lens': tokens_lens
    }

In [35]:
train_dataloader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=0,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

Посмотрим на какой-нибудь батч:

In [36]:
batch = next(iter(train_dataloader))
batch.keys(), batch['targets'], batch['tokens'], batch['tokens_lens']

(dict_keys(['targets', 'texts', 'tokens', 'tokens_lens']),
 tensor([1, 0, 1, 1]),
 tensor([[  30,    5,    2,  225,   87, 4722,    5,    2,    1,    9,   14, 1599,
           183,  175,    2,   88,   84,    9,  199,   10],
         [  10,  132,   14,  363, 1736,   17,   30,  755,  205,  790,    4,   22,
           399,   21,    1,  107,    1,   27,   82, 4706],
         [   6,   28, 3448,    9,   62,   60,  318,   48,   10,   20,   14,   42,
            50,    9,  620,  112,  188,   78,    9,  109],
         [   9, 2073,  548,   10,  380,  183, 1158,  134, 1478,    1,   19,    3,
          3310,    1,  203,   12,   14,  509,    3, 1057]]),
 tensor([20, 20, 20, 20]))

### `Эмбеддинги в PyTorch`

В случае обучение нейронной сети мы будем сами обучать эмбеддинги токенов. Для этого нам нужно использовать класс `torch.nn.Embedding`, который по сути реализует массив.

Эмбеддинг токена $i$ это строка $i$ в матрице эмбеддингов.

- `num_embeddings` - число токенов

- `embedding_dim` - размерность эмбеддинга

- `padding_idx` - индекс токена `<pad>`

```python
torch.nn.Embedding(
    num_embeddings: int,
    embedding_dim: int,
    padding_idx: Optional[int] = None,
    max_norm: Optional[float] = None,
    norm_type: float = 2.0,
    scale_grad_by_freq: bool = False,
    sparse: bool = False,
    _weight: Optional[torch.Tensor] = None,
    _freeze: bool = False,
    device=None,
    dtype=None,
) -> None
```

Посмотрим как преобразуется батч после слоя `nn.Embedding`.

In [37]:
embed_dim = 128

emb_layer = torch.nn.Embedding(
    tokenizer.get_vocab_size(), embed_dim,
    padding_idx=tokenizer.token_to_id('<pad>')
)

In [38]:
out = emb_layer(batch['tokens'])

out.shape

torch.Size([4, 20, 128])

Так как `nn.Embedding` реализует матрицу, то число параметров зависит от размера словаря и размерности эмеддингов.

In [39]:
tokenizer.get_vocab_size() * embed_dim

640000

In [40]:
print_params_count(emb_layer)

Информация о числе параметров модели: Embedding
Всего параметров: 		 640000
Всего обучаемых параметров: 	 640000



### `Нейронная сеть (MLP)`

В этой части мы обучим классификатор текстов на основе полносвязной нейронной сети. Выше мы уже создали удобные класс-обёртки для работы с данными. Теперь мы соберем модель для решения задачи классификации.

In [41]:
class MLPClassifier(torch.nn.Module):
    def __init__(
        self, embedding_dim, hidden_dim, output_size, tokenizer: Tokenizer,
        dropout=0.5
    ):
        super().__init__()

        self.dropout = dropout

        self.tokenizer = tokenizer
        self.hidden_dim = hidden_dim
        self.output_size = output_size
        self.embedding_dim = embedding_dim

        # Create a simple lookup table that stores embeddings of a fixed dictionary and size.
        #    Use torch.nn.Embedding.
        self.word_embeddings = torch.nn.Embedding(
            self.tokenizer.get_vocab_size(), self.embedding_dim,
            padding_idx=self.tokenizer.token_to_id('<pad>')
        )

        self.features = torch.nn.Sequential(
            nn.Linear(self.embedding_dim, self.hidden_dim),
            nn.ReLU(), nn.Dropout1d(self.dropout),
            nn.Linear(self.hidden_dim,    self.hidden_dim),
            nn.ReLU(), nn.Dropout1d(self.dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(), nn.Dropout1d(self.dropout),
        )
        # Create linear layer for classification
        self.output = torch.nn.Sequential(
            torch.nn.Linear(self.hidden_dim, self.output_size),
        )

    def forward(self, tokens):
        """
        :param torch.Tensor(dtype=torch.long) tokens: Batch of texts represented with tokens.
        :return torch.Tensor(dtype=torch.long): Vector representation for each sequence in batch
        """
        # Evaluate embeddings
        embedding = self.word_embeddings(tokens)

        # Make forward pass through MLP network
        h   = embedding.mean(axis=1)
        h   = self.features(h)
        out = self.output(h)

        return out

In [42]:
net = MLPClassifier(
    embedding_dim = 128,
    hidden_dim    = 64,
    output_size   = 2,
    tokenizer     = tokenizer
)

out = net(batch['tokens'])
out.shape

torch.Size([4, 2])

In [43]:
print_params_count(net)

Информация о числе параметров модели: MLPClassifier
Всего параметров: 		 656706
Всего обучаемых параметров: 	 656706



### `Цикл обучения`

In [44]:
set_global_seed(42)

device = 'cpu'

if torch.cuda.is_available():
    device = 'cuda:0'
device

'cuda:0'

Мы описали датасет, даталоадер, архитектуру, следовательно можем переходить к циклу обучения. Мы решаем задачу классификации, поэтому будем использовать `nn.CrossEntropyLoss`

In [45]:
loss_fn = nn.CrossEntropyLoss(reduction='mean')

Для обучения мы будем брать `max_len=100`, то есть оценивать рецензию по первым 100 словам.

In [46]:
max_len    = 100
batch_size = 128
lr         = 3 * 1e-3

embedding_dim = 128
hidden_dim    = 256
output_size   = 2

In [47]:
train_dataset = TextDataset(corpus_train, y_train, tokenizer, max_len=max_len)
test_dataset  = TextDataset(corpus_test , y_train, tokenizer, max_len=max_len)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25000/25000 [00:13<00:00, 1832.92it/s]


In [48]:
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

In [49]:
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

Модель мы определили выше, для оптимизации будем использовать `Adam`.

In [50]:
set_global_seed(42)

net = MLPClassifier(
    embedding_dim = embedding_dim,
    hidden_dim    = hidden_dim,
    output_size   = output_size,
    tokenizer     = tokenizer
)

In [51]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

Цикл обучения сделаем универсальным для разных задач.

In [52]:
@torch.no_grad()
def evaluate(net, valid_dataloader, loss_fn, device):
    """Оценивает производительность сети на валидационном датасете."""
    net.eval()
    total_loss, total_accuracy = 0., 0.
    num_samples = 0

    for batch in valid_dataloader:
        inputs = batch["tokens"].to(device)
        targets = batch["targets"].to(device)

        outputs = net(inputs)
        predictions = torch.argmax(outputs, dim=1)

        # Подсчет средней потери и точности
        batch_size = outputs.size(0)
        total_loss += loss_fn(outputs, targets).item() * batch_size
        total_accuracy += (predictions == targets).sum().item()
        num_samples += batch_size

    avg_loss = total_loss / num_samples
    avg_acc = total_accuracy / num_samples
    return avg_loss, avg_acc


def train_model(epoch_num, net, optimizer, loss_fn, train_dataloader, valid_dataloader, device):
    """Обучение модели и вывод основных метрик в процессе."""
    net = net.to(device)

    for epoch in range(epoch_num):
        running_train_loss = 0.
        running_train_acc = 0.
        num_batches = len(train_dataloader)

        # Режим обучения
        net.train()
        with tqdm(total=num_batches, desc=f'Обучение') as pbar:
            for i, batch in enumerate(train_dataloader):
                optimizer.zero_grad()

                inputs = batch["tokens"].to(device)
                targets = batch["targets"].to(device)

                outputs = net(inputs)
                loss = loss_fn(outputs, targets)
                loss.backward()
                optimizer.step()

                predictions = torch.argmax(outputs, dim=1)
                acc = (predictions == targets).float().mean().item()
                running_train_loss += loss.item()
                running_train_acc += acc

                pbar.set_postfix({'Train Loss': f"{running_train_loss/(i+1):.4f}", 'Train Acc': f"{running_train_acc/(i+1)*100:.2f}%"})
                pbar.update(1)

        # Вычисляем средние значения за эпоху
        avg_train_loss = running_train_loss / num_batches
        avg_train_acc = running_train_acc / num_batches

        # Проверяем, надо ли оценить модель на валидационном датасете
        if (epoch + 1) % 10 == 0 or epoch == epoch_num - 1:
            # Оцениваем на валидационных данных
            val_loss, val_acc = evaluate(net, valid_dataloader, loss_fn, device)

            # Выводим итоговые метрики за эпоху
            print(f"Эпоха {epoch + 1}: Точность на обучении: {avg_train_acc*100:.2f}% | Потеря на обучении: {avg_train_loss:.4f}")
            print(f"Эпоха {epoch + 1}: Точность на валидации: {val_acc*100:.2f}% | Потеря на валидации: {val_loss:.4f}\n")


In [53]:
train_model(
    epoch_num        = 100,
    net              = net,
    optimizer        = optimizer,
    loss_fn          = loss_fn,
    train_dataloader = train_dataloader,
    valid_dataloader = test_dataloader,
    device           = device
)

Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 104.31it/s, Train Loss=0.6532, Train Acc=54.08%]


Эпоха 10: Точность на обучении: 54.08% | Потеря на обучении: 0.6532
Эпоха 10: Точность на валидации: 79.72% | Потеря на валидации: 0.5877



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 105.04it/s, Train Loss=0.6460, Train Acc=53.93%]


Эпоха 20: Точность на обучении: 53.93% | Потеря на обучении: 0.6460
Эпоха 20: Точность на валидации: 78.75% | Потеря на валидации: 0.5666



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 113.85it/s, Train Loss=0.6387, Train Acc=54.78%]


...

Эпоха 80: Точность на обучении: 55.67% | Потеря на обучении: 0.6165
Эпоха 80: Точность на валидации: 77.66% | Потеря на валидации: 0.4808



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 107.75it/s, Train Loss=0.6164, Train Acc=55.57%]


Эпоха 90: Точность на обучении: 55.57% | Потеря на обучении: 0.6164
Эпоха 90: Точность на валидации: 77.60% | Потеря на валидации: 0.4940



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 108.12it/s, Train Loss=0.6144, Train Acc=55.56%]


Эпоха 100: Точность на обучении: 55.56% | Потеря на обучении: 0.6144
Эпоха 100: Точность на валидации: 77.49% | Потеря на валидации: 0.4811



Теперь мы можем пополнить нашу таблицу результатов:

- `word2vec` + `CatBoost` - accuracy 0.79
- `fasttext` + `CatBoost` - accuracy 0.85
- `nn.Embedding` + `MLP`  - accuracy (test) 0.79 (лучшее значение), accuracy (train) ~ 0.55

Важно отметить, что на train мы не смогли получить качество выше 0.60, то есть модель недостаточно выразительна.

**Вопрос:** Как можно повысить качество?

### `Нейронная сеть (CNN)`

Попробуем еще раз посмотреть, что мы именно мы делаем

Общий вид архитектуры:
    
- Переводим токены в вектора фиксированного размера (эмбеддинги)
- Стакаем вектора в матрицу
- Извлекаем признаки из матрицы
- Подаем признаки в логистическую регрессию

Выше мы извлекали признаки не из всей матрицы, а сначала усредняли вдоль строк.

**Вопроc:** Зачем мы усредняли матрицу?

**Вопроc:** Какие способы извлечения признаков из матрицы вы знаете?

Мы можем использовать сверточные архитектуры, только с одной **оговоркой**: мы извлекаем признаки из матрицы, а не из тензора (изображения). Имеено поэтому мы будем использовать `nn.Conv1d` и действовать сверткой вдоль строк (каждая строка отвечает за эмбеддинг своего токена).

**Вопрос:** Что позволяют учитывать свертки?




In [54]:
embed_dim = 128

emb_layer = torch.nn.Embedding(
    tokenizer.get_vocab_size(), embed_dim,
    padding_idx=tokenizer.token_to_id('<pad>')
)

embds = emb_layer(batch['tokens'])
embds.shape

torch.Size([4, 20, 128])

Мы хотим подействовать на строки, поэтому нам нужно транспонировать матрицу эмбеддингов.



In [55]:
embds = embds.transpose(1, 2)
embds.shape

torch.Size([4, 128, 20])

In [56]:
conv_layer = nn.Conv1d(128, 32, kernel_size=3)

out = conv_layer(embds)
out.shape

torch.Size([4, 32, 18])

**Вопрос:** Почему последняя размерность стала 18? Какую информацию хранят новые признаки?

К сожалению, к нам приходят предложения разной длины, но  мы хотим уметь работать с предложениями произвольной длины.

**Вопрос:** Как мы обрабатывали изображения произвольного размера? Как надо модифицировать архитектуру?

In [57]:
avg_pool_global = nn.AdaptiveAvgPool1d(output_size=1)

out_pool = avg_pool_global(out)
out_pool.shape

torch.Size([4, 32, 1])

На основе этой идеи построим модель.

In [58]:
class CNNClassifier(torch.nn.Module):
    def __init__(
        self, embedding_dim, hidden_dim, output_size, tokenizer: Tokenizer,
        dropout=0.5
    ):
        super().__init__()

        self.dropout = dropout

        self.tokenizer = tokenizer
        self.hidden_dim = hidden_dim
        self.output_size = output_size
        self.embedding_dim = embedding_dim

        # Create a simple lookup table that stores embeddings of a fixed dictionary and size.
        #    Use torch.nn.Embedding.
        self.word_embeddings = torch.nn.Embedding(
            self.tokenizer.get_vocab_size(), self.embedding_dim,
            padding_idx=self.tokenizer.token_to_id('<pad>')
        )

        self.features = torch.nn.Sequential(
            nn.Conv1d(self.embedding_dim, self.hidden_dim, 3), # Поменяли
            nn.ReLU(), nn.Dropout1d(self.dropout),
            nn.Conv1d(self.hidden_dim,    self.hidden_dim, 3), # Поменяли
            nn.ReLU(), nn.Dropout1d(self.dropout),
            nn.AdaptiveAvgPool1d(output_size=1),               # Поменяли
            nn.Flatten(start_dim=1)                            # Поменяли
        )
        # Create linear layer for classification
        self.output = torch.nn.Sequential(
            torch.nn.Linear(self.hidden_dim, self.output_size),
        )

    def forward(self, tokens):
        """
        :param torch.Tensor(dtype=torch.long) tokens: Batch of texts represented with tokens.
        :return torch.Tensor(dtype=torch.long): Vector representation for each sequence in batch
        """
        # Evaluate embeddings
        embedding = self.word_embeddings(tokens)

        # Make forward pass through CNN network
        h   = embedding.transpose(1, 2)                        # Поменяли
        h   = self.features(h)
        out = self.output(h)

        return out

In [59]:
net = CNNClassifier(
    embedding_dim = 128,
    hidden_dim    = 64,
    output_size   = 2,
    tokenizer     = tokenizer
)

out = net(batch['tokens'])
out.shape

torch.Size([4, 2])

In [60]:
print_params_count(net)

Информация о числе параметров модели: CNNClassifier
Всего параметров: 		 677122
Всего обучаемых параметров: 	 677122



### `Обучение CNN`

В случае сверточных нейронных сетей мы можем достаточно быстро переобучиться, поэтому уменьшим число параметров модели.

In [61]:
max_len    = 100
batch_size = 128
lr         = 3 * 1e-3

embedding_dim = 32
hidden_dim    = 16
output_size   = 2

In [62]:
set_global_seed(42)

In [63]:
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

In [64]:
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

In [65]:
set_global_seed(42)

net = CNNClassifier(
    embedding_dim = embedding_dim,
    hidden_dim    = hidden_dim,
    output_size   = output_size,
    tokenizer     = tokenizer
)

In [66]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

In [67]:
train_model(
    epoch_num        = 100,
    net              = net,
    optimizer        = optimizer,
    loss_fn          = loss_fn,
    train_dataloader = train_dataloader,
    valid_dataloader = test_dataloader,
    device           = device
)

Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 102.43it/s, Train Loss=0.3987, Train Acc=81.99%]


Эпоха 10: Точность на обучении: 81.99% | Потеря на обучении: 0.3987
Эпоха 10: Точность на валидации: 79.59% | Потеря на валидации: 0.4411



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 104.37it/s, Train Loss=0.3414, Train Acc=85.39%]


Эпоха 20: Точность на обучении: 85.39% | Потеря на обучении: 0.3414
Эпоха 20: Точность на валидации: 77.37% | Потеря на валидации: 0.5004



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 115.62it/s, Train Loss=0.2842, Train Acc=88.38%]


...

Эпоха 80: Точность на обучении: 95.04% | Потеря на обучении: 0.1471
Эпоха 80: Точность на валидации: 72.14% | Потеря на валидации: 1.3897



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 113.06it/s, Train Loss=0.1462, Train Acc=95.10%]


Эпоха 90: Точность на обучении: 95.10% | Потеря на обучении: 0.1462
Эпоха 90: Точность на валидации: 71.75% | Потеря на валидации: 1.4581



Обучение: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:01<00:00, 106.25it/s, Train Loss=0.1365, Train Acc=95.39%]


Эпоха 100: Точность на обучении: 95.39% | Потеря на обучении: 0.1365
Эпоха 100: Точность на валидации: 71.80% | Потеря на валидации: 1.5596



При обучении нейронных сетей на маленьких выборках можно очень просто переобучить модель. Заметим, что на train мы смогли добиться почти идеального качества, значит, модель извлекает достаточно информации о выборке.

Мы бы могли повысить качество и уменьшить переобучение следующим образом:

- Взять в качестве инициализации эмбеддингов FastText.
- Добавить больше регуляризации для обучения.
- Использовать более умную токенизацию.
- Лучше предобработать текст.


Однако при прочих равных на нашем модельном эксперименте получились следующие результаты:
    

- `word2vec` + `CatBoost` - accuracy 0.79
- `fasttext` + `CatBoost` - accuracy 0.85
- `nn.Embedding` + `MLP`  - accuracy (test) 0.79, accuracy (train) ~ 0.55
- `nn.Embedding` + `CNN`  - accuracy (test) 0.79 (лучшее значение), accuracy (train) ~ 0.95

**Вопрос:** Какие смысловые проблемы вы видете в сверточных сетях?

## `Выводы`

- Словарь отражает ваше знание о задаче, вы можете заложить некоторый `Inductive bias` в словарь. Например, если у вас задача генерации молекул, то можно создать словарь из которого гарантировано будут получаться хорошие молекулы (синтетически доступные).
- Использовать стандартные приёмы для получения эмбеддингов бывает полезно как в качестве **теоретического бейзлайна**, так и на **практике**.
- Можно задавать различные архитектуры: полносвязные, сверточные.
- Мы пока не научились учитывать контекст произвольной длины.